In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

from IPython.display import display

PROJECT_ROOT = Path.cwd().parent

print("Project root:", PROJECT_ROOT)

Project root: d:\Research_Project\sales-forecasting-platform


In [2]:
# declare paths for raw and processed data
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

RAW_SALES_FILE = RAW_DATA_DIR / "online_retail_II.xlsx"

print("Raw file:", RAW_SALES_FILE)
print("Processed directory:", PROCESSED_DATA_DIR)

Raw file: d:\Research_Project\sales-forecasting-platform\data\raw\online_retail_II.xlsx
Processed directory: d:\Research_Project\sales-forecasting-platform\data\processed


In [3]:
# check if the raw sales data file exists
if not RAW_SALES_FILE.exists():
    raise FileNotFoundError(
        f"Dataset not found: {RAW_SALES_FILE}"
    )

print("Dataset exists:", RAW_SALES_FILE)

Dataset exists: d:\Research_Project\sales-forecasting-platform\data\raw\online_retail_II.xlsx


In [4]:
# create folder for processed data if it doesn't exist
PROCESSED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Processed directory ready:")
print(PROCESSED_DATA_DIR)

Processed directory ready:
d:\Research_Project\sales-forecasting-platform\data\processed


In [5]:
# read the raw sales data from the Excel file
df = pd.read_excel(
    RAW_SALES_FILE,
    sheet_name=0
)

print("Raw shape:", df.shape)

display(df.head())

Raw shape: (525461, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [6]:
excel_file = pd.ExcelFile(RAW_SALES_FILE)

print("Available sheets:")
print(excel_file.sheet_names)

Available sheets:
['Year 2009-2010', 'Year 2010-2011']


In [9]:
# read the raw sales data from the Excel file
df_2009_2010 = pd.read_excel(
    RAW_SALES_FILE,
    sheet_name=0
)

df_2010_2011 = pd.read_excel(
    RAW_SALES_FILE,
    sheet_name=1
)

df = pd.concat(
    [
        df_2009_2010,
        df_2010_2011
    ],
    ignore_index=True
)

print("Combined shape:", df.shape)

Combined shape: (1067371, 8)


In [10]:
working_df = df.copy()

print("Working dataframe shape:", working_df.shape)

Working dataframe shape: (1067371, 8)


In [11]:
print(working_df.columns.tolist())

['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


In [12]:
display(
    working_df[
        [
         'Invoice', 
         'StockCode', 
         'Description', 
         'Quantity', 
         'InvoiceDate',
         'Price',
         'Customer ID',
        'Country'
        ]

    ].head(10)
)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom
8,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,13085.0,United Kingdom


In [13]:
working_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 65.1+ MB


In [14]:
working_df["InvoiceDate"] = pd.to_datetime(
    working_df["InvoiceDate"],
    errors="coerce"
)

print(
    "Invalid InvoiceDate:",
    working_df["InvoiceDate"].isna().sum()
)

Invalid InvoiceDate: 0


In [15]:
invoice_as_text = (
    working_df["Invoice"]
    .astype("string")
    .str.strip()
)
working_df["Invoice"] = invoice_as_text

working_df["is_cancelled"] = (
    working_df["Invoice"]
    .str.upper()
    .str.startswith("C")
)

print(
    working_df["is_cancelled"]
    .value_counts()
)

is_cancelled
False    1047877
True       19494
Name: count, dtype: Int64


In [16]:
working_df["line_revenue"] = (
    working_df["Quantity"]
    * working_df["Price"]
)

display(
    working_df[
        [
            "Invoice",
            "Quantity",
            "Price",
            "line_revenue",
            "is_cancelled"
        ]
    ].head(20)
)

,Invoice,Quantity,Price,line_revenue,is_cancelled
0,489434,12,6.95,83.40,False
1,489434,12,6.75,81.00,False
2,489434,12,6.75,81.00,False
3,489434,48,2.10,100.80,False
4,489434,24,1.25,30.00,False
5,489434,24,1.65,39.60,False
6,489434,24,1.25,30.00,False
7,489434,10,5.95,59.50,False
8,489435,12,2.55,30.60,False
9,489435,12,3.75,45.00,False


In [17]:
negative_revenue = working_df[
    working_df["line_revenue"] < 0
]

print(
    "Negative revenue rows:",
    len(negative_revenue)
)

print(
    "Negative revenue total:",
    negative_revenue["line_revenue"].sum()
)

Negative revenue rows: 19498
Negative revenue total: -1685717.57


In [18]:
cancel_summary = (
    working_df
    .groupby("is_cancelled")["line_revenue"]
    .agg(
        rows="count",
        total_revenue="sum",
        average_revenue="mean",
        minimum_revenue="min"
    )
)

display(cancel_summary)

,rows,total_revenue,average_revenue,minimum_revenue
is_cancelled,,,,
False,1047877,2.081392e+07,19.862940,-53594.36
True,19494,-1.526668e+06,-78.314756,-168469.60
